<a href="https://colab.research.google.com/github/Yusef-Hazem/Lang-Frameworks/blob/main/04_indexing_to_vectorDBs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Langchain/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Langchain


In [ ]:
# #Python 3.12.13
!pip install -U langchain==1.3.11
!pip install requests==2.34.2
!pip install -U langchain_community==0.4.2
!pip install -U python-dotenv==1.2.2
!pip install -U langchain-huggingface==1.2.2
!pip install faiss-cpu==1.14.3
!pip install langchain-groq==1.1.3
!pip install chromadb

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import bs4
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

True

Multi-Representaion index

In [ ]:
docs = WebBaseLoader(["https://www.coursera.org/learn/introduction-to-retrieval-augmented-generation-rag" ,
                      "https://aws.amazon.com/what-is/retrieval-augmented-generation/"]).load()

In [ ]:
llm= ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
template  = """Summarrize the following document : \n\n {document}"""
prompt = ChatPromptTemplate.from_template(template)
chain =({"document" :  (lambda x : x.page_content)} |prompt | llm | StrOutputParser())


# use .batch to get more than 1 input , and can excute concurrency
summaries = chain.batch(docs , {"max_concurrency ":1})


In [ ]:
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.stores import InMemoryByteStore
from langchain_classic.retrievers import MultiVectorRetriever
import uuid
from langchain_core.documents import Document


In [ ]:
# make multi-vectorRetriever to use one to search and another for retrival
vector_store = Chroma(collection_name= "summaries" , embedding_function=embedding_model) # this for search
store = InMemoryByteStore() # this for retriver

id_key = "doc_id"
retriver = MultiVectorRetriever(
    vectorstore=vector_store,
    byte_store= store,
    id_key = id_key)

# generate uniqe id for each doc loaded

doc_ids = [str(uuid.uuid4()) for _ in docs ]



summary_docs = [
    Document(page_content= d , metadata= {id_key : doc_ids[i] })
    for i , d  in enumerate(summaries)
]

# add the summary docs&metadata to the vdb
retriver.vectorstore.add_documents(summary_docs)

#add the the original docs to the store -InMemoryByteStore- to retrive the original doc

retriver.docstore.mset(tuple(zip(doc_ids , docs)))


/tmp/ipykernel_16227/4219347536.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(collection_name= "summaries" , embedding_function=embedding_model) # this for search


In [ ]:
summary_docs[0].metadata , doc_ids[0]

({'doc_id': 'ccd079c6-8cfd-4f84-8c22-d96f1245d5cd'},
 'ccd079c6-8cfd-4f84-8c22-d96f1245d5cd')

In [ ]:
docs[0]

Document(metadata={'source': 'https://www.coursera.org/learn/introduction-to-retrieval-augmented-generation-rag', 'title': 'Introduction to Retrieval Augmented Generation (RAG) | Coursera', 'description': 'Offered by Coursera. In this course, we start with the concepts and use of Large Language Models, exploring popular LLMs such as OpenAI GPT ... Enroll for free.', 'language': 'en'}, page_content='Introduction to Retrieval Augmented Generation (RAG) | Coursera\n\n\n\n\n\n\nFor IndividualsFor BusinessesFor UniversitiesFor GovernmentsExploreDegrees\u200bLog InJoin for FreeJoin for FreeIntroduction to Retrieval Augmented Generation (RAG)AboutOutcomesModulesTestimonialsReviewsPreviousNextBrowseInformation TechnologySupport and Operations Ends soon! Keep adding new skills with 10,000+ programs for $239 (usually $399). Save now.Introduction to Retrieval Augmented Generation (RAG)This course is part of Building GenAI Applications and Agents SpecializationInstructors: Manas Dasgupta +1 moreEn

In [ ]:
query = "what is Rag ?"
# this serches in the summaries -vectordatabase-
vector_store.similarity_search(query)

[Document(metadata={'doc_id': 'ccd079c6-8cfd-4f84-8c22-d96f1245d5cd'}, page_content='The document is about an online course on "Introduction to Retrieval Augmented Generation (RAG)" offered on Coursera. Here\'s a summary:\n\n**Course Overview**\n\nThe course is part of the "Building GenAI Applications and Agents Specialization" and is designed to introduce students to Retrieval Augmented Generation (RAG) and its applications. RAG is a technology that combines the powers of Large Language Models (LLMs) and LLM frameworks to create automated workflows and applications.\n\n**Course Objectives**\n\nBy the end of the course, students will be able to:\n\n* Demonstrate Large Language Model capabilities in Natural Language based Automations\n* Develop RAG applications using LLM frameworks, models, and vector databases\n* Use vector databases as a storage medium for language embeddings in RAG applications\n* Write effective prompts and understand models and tokens\n\n**Course Content**\n\nThe c

In [ ]:
# this searches in summaries and get the orginal document from docstore
retriver.invoke(query)

[Document(metadata={'source': 'https://www.coursera.org/learn/introduction-to-retrieval-augmented-generation-rag', 'title': 'Introduction to Retrieval Augmented Generation (RAG) | Coursera', 'description': 'Offered by Coursera. In this course, we start with the concepts and use of Large Language Models, exploring popular LLMs such as OpenAI GPT ... Enroll for free.', 'language': 'en'}, page_content='Introduction to Retrieval Augmented Generation (RAG) | Coursera\n\n\n\n\n\n\nFor IndividualsFor BusinessesFor UniversitiesFor GovernmentsExploreDegrees\u200bLog InJoin for FreeJoin for FreeIntroduction to Retrieval Augmented Generation (RAG)AboutOutcomesModulesTestimonialsReviewsPreviousNextBrowseInformation TechnologySupport and Operations Ends soon! Keep adding new skills with 10,000+ programs for $239 (usually $399). Save now.Introduction to Retrieval Augmented Generation (RAG)This course is part of Building GenAI Applications and Agents SpecializationInstructors: Manas Dasgupta +1 moreE

ColBERT

In [ ]:
# here we want to call pretrained model to works as colbert -embed by token-
# needed to change lanchain version to be compatable
!pip uninstall -y transformers
!pip install "transformers==4.37.2"
!pip install "langchain==0.1.20"
!pip install "ragatouille==0.0.8"

Found existing installation: transformers 4.37.2
Uninstalling transformers-4.37.2:
  Successfully uninstalled transformers-4.37.2
  Using cached transformers-4.37.2-py3-none-any.whl.metadata (129 kB)
Using cached transformers-4.37.2-py3-none-any.whl (8.4 MB)


In [ ]:
from ragatouille import RAGPretrainedModel
colbert_model = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

artifact.metadata: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[Jul 02, 23:48:50] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
docs = WebBaseLoader(["https://www.coursera.org/learn/introduction-to-retrieval-augmented-generation-rag" ,
                      "https://aws.amazon.com/what-is/retrieval-augmented-generation/"]).load()

In [ ]:
docs_pagecontent = [doc.page_content for doc in docs ]
colbert_model.index(collection=docs_pagecontent)

No index_name received! Using default index_name (colbert-ir/colbertv2.0_new_index)
---- WARNING! You are using PLAID with an experimental replacement for FAISS for greater compatibility ----
This is a behaviour change from RAGatouille 0.8.0 onwards.
This works fine for most users and smallish datasets, but can be considerably slower than FAISS and could cause worse results in some situations.
If you're confident with FAISS working on your machine, pass use_faiss=True to revert to the FAISS-using behaviour.
--------------------


[Jul 02, 23:52:30] #> Creating directory .ragatouille/colbert/indexes/colbert-ir/colbertv2.0new_index 




/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Jul 02, 23:52:32] [0] 		 #> Encoding 25 passages..


/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


INFO: Tried to clean up context but failed!


...


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(

100%|██████████| 1/1 [00:23<00:00, 23.30s/it]
1it [00:23, 23.42s/it]
100%|██████████| 1/1 [00:00<00:00, 382.31it/s]

[Jul 02, 23:53:21] #> Optimizing IVF to store map from centroids to list of pids..
[Jul 02, 23:53:21] #> Building the emb2pid mapping..
[Jul 02, 23:53:21] len(emb2pid) = 4665



100%|██████████| 1024/1024 [00:00<00:00, 62724.06it/s]

[Jul 02, 23:53:21] #> Saved optimized IVF to .ragatouille/colbert/indexes/colbert-ir/colbertv2.0new_index/ivf.pid.pt
Done indexing!


'.ragatouille/colbert/indexes/colbert-ir/colbertv2.0new_index'

In [ ]:
colbert_model.search(query = "how rage works")

Loading searcher for index colbert-ir/colbertv2.0new_index for the first time... This may take a few seconds


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Jul 02, 23:54:38] #> Loading codec...
[Jul 02, 23:54:38] #> Loading IVF...
[Jul 02, 23:54:38] Loading segmented_lookup_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


[Jul 02, 23:55:10] #> Loading doclens...


...


100%|██████████| 1/1 [00:00<00:00, 342.92it/s]

[Jul 02, 23:55:10] Loading filter_pids_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


[Jul 02, 23:55:41] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
Searcher loaded!

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: . how rage works, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([ 101,    1, 2129, 7385, 2573,  102,  103,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103])
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])



/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


[{'content': "What is Retrieval-Augmented Generation?\n\n\n\nRetrieval-Augmented Generation (RAG) is the process of optimizing the output of a large language model, so it references an authoritative knowledge base outside of its training data sources before generating a response. Large Language Models (LLMs) are trained on vast volumes of data and use billions of parameters to generate original output for tasks like answering questions, translating languages, and completing sentences. RAG extends the already powerful capabilities of LLMs to specific domains or an organization's internal knowledge base, all without the need to retrain the model. It is a cost-effective approach to improving LLM output so it remains relevant, accurate, and useful in various contexts.\n\n\n\n\n\n\n\nWhy is Retrieval-Augmented Generation important?\n\n\n\nLLMs are a key artificial intelligence (AI) technology powering intelligent chatbots and other natural language processing (NLP) applications. The goal is

In [ ]:
retriever = colbert_model.as_langchain_retriever(k=3)
retriever.invoke("What is an example for form based indexing?")

/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


[Document(page_content='Create external data\nThe new data outside of the LLM\'s original training data set is called external data. It can come from multiple data sources, such as a APIs, databases, or document repositories. The data may exist in various formats like files, database records, or long-form text. Another AI technique, called embedding language models, converts data into numerical representations and stores it in a vector database. This process creates a knowledge library that the generative AI models can understand.\nRetrieve relevant information\nThe next step is to perform a relevancy search. The user query is converted to a vector representation and matched with the vector databases. For example, consider a smart chatbot that can answer human resource questions for an organization. If an employee searches, "How much annual leave do I have?" the system will retrieve annual leave policy documents alongside the individual employee\'s past leave record. These specific doc